In [0]:
-- Fetch the qdp_locations_all document data product created by 01. Quantexa Processing

-- 01. Create variables for use in this notebook
-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;



DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;




DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;




-- 2. Remove all Address records for the Tennant
DELETE FROM demo_banking_silver.qdp_locations_all.sat_address s
WHERE s.address_id IN (
  SELECT DISTINCT address_id
  FROM demo_banking_silver.qdp_locations_all.hub_address
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_silver.qdp_locations_all.hub_address WHERE tennant_id = :p_tennant_id;


-- Insert into hub_address
INSERT INTO demo_banking_silver.qdp_locations_all.hub_address
(address_entity_id, tennant_id, period_start, period_end)
SELECT address_entity_id, tennant_id, period_start, period_end
FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_locations_all;

-- Insert into sat_address
INSERT INTO demo_banking_silver.qdp_locations_all.sat_address
    (sat_address_id, 
     address_id, 
     load_datetime, 
     record_source_id, 
     postal_code,
     data_source_code,
     data_source_concept_id,
     data_source_raw_code,
     data_source_raw_concept_id,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     flatnumber_housenumber_housename,
     house_number,
     house_name,
     building_number,
     building_name,
     address_line1,
     address_line2,
     address_line3,
     address_line4,
     address_line5,
     address_line6,
     address_line7,
     address_line8,
     street,
     district,
     city,
     county,
     state,
     country,
     country_code,
     country_concept_id,
     country_raw_code,
     country_raw_concept_id,
     full_address,
     uprn,
     what3words,
     latitude,
     longitude,
     directions,
     period_start,
     period_end
    )
SELECT
  monotonically_increasing_id() AS sat_address_id,
  h.address_id AS address_id,
  run_timestamp AS load_datetime,
  try_cast(1 AS BIGINT) AS record_source_id,
  q.postal_code AS postal_code,
  q.data_source_code AS data_source_code,
  q.data_source_concept_id AS data_source_concept_id,
  q.data_source_raw_code AS data_source_raw_code,
  q.data_source_raw_concept_id AS data_source_raw_concept_id,
  q.type_code AS type_code,
  q.type_concept_id AS type_concept_id,
  q.type_raw_code AS type_raw_code,
  q.type_raw_concept_id AS type_raw_concept_id,
  q.flatnumber_housenumber_housename AS flatnumber_housenumber_housename,
  q.house_number AS house_number,
  q.house_name AS house_name,
  q.building_number AS building_number,
  q.building_name AS building_name,
  q.address_line1 AS address_line1,
  q.address_line2 AS address_line2,
  q.address_line3 AS address_line3,
  q.address_line4 AS address_line4,
  q.address_line5 AS address_line5,
  q.address_line6 AS address_line6,
  q.address_line7 AS address_line7,
  q.address_line8 AS address_line8,
  q.street AS street,
  q.district AS district,
  q.city AS city,
  q.county AS county,
  q.state AS state,
  q.country AS country,
  q.country_code AS country_code,
  q.country_concept_id AS country_concept_id,
  q.country_code_raw AS country_code_raw,
  q.country_raw_concept_id AS country_raw_concept_id,
  q.full_address AS full_address,
  q.uprn AS uprn,
  q.what3words AS what3words,
  try_cast(q.latitude AS DOUBLE) AS latitude,
  try_cast(q.longitude AS DOUBLE) AS longitude,
  q.directions AS directions,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_locations_all q
JOIN demo_banking_silver.qdp_locations_all.hub_address h
  ON h.address_entity_id = q.address_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_locations_all.sat_address;
